# Pregunta 1 — Diferència entre la nota del públic i de la crítica, per pel·lícula i gènere
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` i `actors` en llistes (cada pel·lícula en pot tenir diversos).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

df.loc[df["revenue"] == 0, "revenue"] = np.nan

df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])
df["actors_list"] = df["actors"].str.split(",").apply(lambda l: [a.strip() for a in l])

df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Quines pel·lícules tenen més diferència entre la nota del públic i de la crítica, i quins gèneres són més propensos a aquesta diferència?**

**Mètode:** descartem les pel·lícules sense `metascore`. Definim `diferencia = metascore − rating×10` (ambdues notes passades a escala 0–100): un valor negatiu vol dir que la crítica puntua més baix que el públic, i un de positiu, més alt. Ordenem per valor absolut per trobar les pel·lícules més extremes, i calculem la diferència mitjana per gènere (explotant la llista de gèneres).

In [ ]:
sub = df.dropna(subset=["metascore"]).copy()
sub["diferencia"] = sub["metascore"] - sub["rating"] * 10
print(f"Pel·lícules amb metascore disponible: {len(sub)} de {len(df)}")
print(f"Diferència mitjana: {sub['diferencia'].mean():.2f} (desviació típica: {sub['diferencia'].std():.2f})")

### Pel·lícules amb més diferència

In [ ]:
top_dif = sub.reindex(sub["diferencia"].abs().sort_values(ascending=False).index).head(10)
top_dif[["title", "year", "rating", "metascore", "diferencia"]]

### Diferència mitjana per gènere

In [ ]:
g = sub.explode("genres")
per_genere = (g.groupby("genres")
                .agg(n=("title", "count"), diferencia_mitjana=("diferencia", "mean"))
                .query("n >= 20")
                .sort_values("diferencia_mitjana")
                .round(2))
per_genere

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
per_genere["diferencia_mitjana"].plot(kind="barh", color="steelblue", ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Metascore − Rating×10 mitjà, per gènere")
ax.set_xlabel("Diferència (punts sobre 100)")
plt.tight_layout(); plt.show()

## 3. Resultats

- Les diferències més grans es donen en ambdues direccions: **Collateral Beauty** (6,8 públic / 23 crítica, diferència −45), **Taare Zameen Par** (8,5 públic / 42 crítica, −43) i **Dirty Grandpa** (6,0 / 18, −42) tenen la crítica molt per sota del públic; **Megan Is Missing** (4,9 públic / 94 crítica, +45) és el cas contrari més extrem.
- La diferència mitjana global és d'uns **−8 punts**: de mitjana, la crítica puntua una mica més baix que el públic.
- Per gènere, **Acció** (−12,2), **Romance** (−11,5) i **Família** (−10,9) són on la crítica és relativament més dura; **Història** (−0,3) i **Animació** (−2,6) són on més s'ajusten ambdues notes.


## 4. Interpretació i limitacions

Té sentit que Acció i Romance siguin els gèneres amb més discrepància: prioritzen l’entreteniment per sobre de l’originalitat que valora la crítica. Historia i Animació s’ajusten més perquè solen ser projectes cuidats que agraden a tothom.